# Debugging with Nsight Visual Studio Code Edition

## Overview

In this section you'll get hands-on experience with debugging Numba CUDA kernels with Nsight Visual Studio Code Edition.

## Learning Objectives

This section will teach you how to:

1. Control program execution through operations such as `Step Over`, `Step Into`, and `Continue`.
2. View variable values and their types in the variable window.
3. Set breakpoints either by line number or by inserting a conditional breakpoint into the program source code.

In this exercise, we will use a modified `sharpen1.py` program from the Nsight Compute lab. A bug has been inserted into this kernel for you to find and fix.

## Starting Nsight Visual Studio Code Edition

To start debugging with Nsight Visual Studio Code Edition in your browser window, run the following cell to create a custom `Open Nsight Visual Studio Code Edition` link. Then click on that link to open the debugger graphical user interface.

In [ ]:
# Run this cell to create the Nsight Visual Studio Code launch button
from IPython.display import HTML, display

# Button that opens a runtime-determined link (hostname from browser at click time)
display(HTML("""
<style>
  .vscode-open-btn {
    font-size: 1.25em;
    font-weight: bold;
    padding: 10px 20px;
    color: white;
    background-color: #0078d4;
    border: none;
    border-radius: 4px;
    cursor: pointer;
  }
  .vscode-open-btn:hover { background-color: #106ebe; }
</style>
<center>
  <button class="vscode-open-btn" type="button"
    onclick="window.open('http://' + window.location.hostname + '/code-server/?workspace=/dli/task/debugging/exercise1/debugging.code-workspace', '_blank', 'noopener,noreferrer');">
    Open Nsight Visual Studio Code Edition
  </button>
</center>
"""))

### Navigation Controls

When first opened, VSCode starts with the Explorer view, which displays the directory tree for your workspace. The following image shows the icons on the left sidebar that you'll use to switch between different views.

<img src="images/vscode-sidebar-controls.png">

## Running the Program for the First Time

Open the example source code by clicking on the `sharpen1.py` file listed on the left side in the Explorer pane.

From skimming through `sharpen_kernel()`, we can see that the pixels are:

1. Scaled from their input range of `0..255` to a range of `0..1`.
2. Converted from `RGB` to `HSV` space by calling `rgb_to_hsv()`.
3. Some operations are performed in `HSV` space based on the pixels' neighbors.
4. The pixels are then converted back to `RGB` space by a call to `hsv_to_rgb()`.

The first step is to run the program to completion to generate the flawed sharpened output image.

Click on the `Run and Debug` icon to switch to the debugging view. There will be a drop-down menu near the top left corner with a list of available debugging configurations. Select the `Exercise 1: Sharpen Image Kernel` menu item.

<img src="images/vscode-sharpen1.png"/>

To start the program running under the debugger, click on the green `Run` arrow icon next to the drop-down menu, or press `F5`. The program will load the initial image from disk and launch the sharpen kernel such that each pixel has its own thread. After the kernel completes, the sharpened image is then written back out to disk. At this point, you should see the following output text in the `Debug Console` at the bottom of the window:

Now that the sharpened image has been created, run the following cell to view both the blurry and sharpened versions of the test image. This will also create a `Refresh Images` button that will reload the images, allowing you to examine the output as you debug and make changes to the code.

It should be easy to see that there's something wrong with the output image. This is the bug we're going to find and fix.

In [ ]:
# Run this cell to create the image comparison widget
import base64
from pathlib import Path
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets

blur_path = Path("/dli/task/debugging/exercise1/images/tabby_tiger_cat_blurred.jpg")
sharp_path = Path("/dli/task/debugging/exercise1/tabby_tiger_cat_sharpened.png")

def img_tag(path, mime_type, width):
    if not path.exists():
        return f"<div>Missing image: {path}</div>"
    b64 = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"<center><img src=\"data:{mime_type};base64,{b64}\" width=\"{width}\"></center>"


def render():
    before = img_tag(blur_path, "image/jpeg", 512)
    after = img_tag(sharp_path, "image/png", 512)
    html = f"""
    <style>
      table.sharpened th {{ font-size: 20px; font-weight: 700; }}
    </style>
    <table class="sharpened" style="margin-left: 0;">
      <tr><th><center>Before Sharpening</center></th><th><center>After Sharpening</center></th></tr>
      <tr>
        <td>{before}</td>
        <td>{after}</td>
      </tr>
    </table>
    """
    with out:
        clear_output(wait=True)
        display(HTML(html))

out = widgets.Output()

btn = widgets.Button(description="Refresh Images")
btn.add_class("refresh-images-btn")
btn.on_click(lambda _=None: render())

display(widgets.HTML("""
<style>
/* Prevent the widget container from clipping the button */
.jupyter-widgets div.refresh-images-btn {
  overflow: visible !important;
  min-height: 46px !important;
}
button.refresh-images-btn,
.jupyter-widgets button.refresh-images-btn {
  font-size: 1.25em !important;
  font-weight: 700 !important;
  padding: 12px 24px !important;
  color: white !important;
  background-color: #0078d4 !important;
  border: none !important;
  border-radius: 4px !important;
  min-width: fit-content;
  min-height: 46px !important;
  line-height: 1.2 !important;
  box-sizing: border-box !important;
}
button.refresh-images-btn:hover,
.jupyter-widgets button.refresh-images-btn:hover {
  background-color: #106ebe !important;
}
</style>
"""))

btn_box = widgets.HBox([btn], layout=widgets.Layout(width="100%", justify_content="center"))
display(out, btn_box)
render()

## Line Number Breakpoints

To debug this problem we begin by using a line number breakpoint to stop program execution at the conversion from `RGB` space to `HSV` space.

Using the editor window, scroll down to line `66` in the `sharpen1.py` file and click to the left of the line number to set a breakpoint. This is the call site for the initial translation of the pixel's `RGB` values to `HSV` space using `rgb_to_hsv()`. The breakpoint will be indicated by a red dot next to the line number.

Use the `Run` button to re-run the program until the breakpoint is reached. At this point we can examine the inputs and outputs of the `rgb_to_hsv()` function.

<img src="images/vscode-first-breakpoint.png"/>

It's not important what the coordinates of the pixel we're examining are - all of the pixels in the output image appear to have the same problem. But it is important that we compare the before and after `RGB` values for the **same** pixel (which means for the same thread).

In this example, the selected pixel is located at `(x=8, y=12)`, with RGB values `(112, 111, 109)`. These `r`, `g`, and `b` values appear reasonable as each is in the expected range of `0..255`.

Now `Step Over` the call and examine the values returned. The `h_orig`, `s_orig`, and `v_orig` values look plausible as `h` is in degrees and is between `0..360`, and `s` and `v` are in the range of `0..1`. So the problem is possibly somewhere later in the code.

The values you see for variables in your Variables panel are likely different from the above due to the way that warps can execute on the GPU independently from one another. This influences which thread reaches the breakpoint first. You will need your values for the `x`, `y`, and `RGB` variables in the next steps in order to compare the before and after `RGB` values.

## Conditional Breakpoints

Next, we'll check the conversion from `HSV` back to `RGB` at the end of the sharpen kernel for the same pixel.

For this step, we'll insert a conditional breakpoint into the code. Scroll down to around line `99` and uncomment the following two lines of code so that the source looks like the following. Make sure to substitute the coordinates of the pixel from earlier for the constants `8` and `12` in the code below.

<img src="images/vscode-conditional-breakpoint.png"/>

Breakpoints set on line numbers stop the program when the first thread reaches them, whereas conditional breakpoints can be more selective about which thread(s) call the `breakpoint()` function.

Since we modified the source code, we need to restart the program for the changes to take effect. This only requires a few simple steps:

1. Click on the red dot next to line `66` to clear the breakpoint you set earlier. Otherwise, this breakpoint will be hit first before the line of code we're debugging.
2. Click on the `Restart` button in the run-control panel at the top center of the window. This forces the program you're debugging to exit and a new instance is started.

Only the thread with the pixel coordinates we specified above will take the `if` branch and make the `breakpoint()` call. This focuses the debugger on the same pixel (thread) that we examined earlier. At this point the program should be stopped on line `102`, which is the next executable line of code.

Now use the `Step Over` button to step over the call to `hsv_to_rgb()` so that we can examine its return values.

<img src="images/vscode-hsv_to_rgb.png"/>

Notice that many of the variables are displayed as `<optimized out>`. This happens when the compiler optimizes away variables that are no longer used, which is common near the end of a function like this. `range_weight` is one such variable which was in use previously but is no longer needed at this point in the code.

Note that the values for `r_out`, `g_out`, and `b_out` will not be identical to what we saw earlier as they've been modified by the sharpening process. However, we can still sanity-check their values against what was originally passed in.

The value for `r_out` looks correct (it was `112` earlier), but the `g_out` and `b_out` values don't appear to hold reasonable values - both are far smaller in magnitude than either `r_out` calculated here or the original input values of `(112, 111, 109)`. Since the original input `RGB` values were first scaled from `0..255` to `0..1`, this is starting to look like a simple scaling problem.

We are now close to isolating the line of code with the bug. It appears that the `rgb_to_hsv()` call is returning reasonable results, but `hsv_to_rgb()` does not appear to be. Working backwards, the next piece of code to examine is the return statement at line `54`. Set a breakpoint on the return statement using one of the techniques described above and continue to debug until you've isolated and fixed the bug.

## Hint

If you're not confident about how to proceed from there, the following cell describes the next steps in the debugging process. Expand the cell to read more.

Disable or remove any other breakpoints you've set, and re-run the program using the `Restart` button. You can then examine the `r`, `g`, and `b` values at the point of the return statement and compare them with what is actually returned to the caller. We've already determined that the return values from `hsv_to_rgb()` are likely incorrect; we just need to work backwards from here to find the incorrect line of code.

Again: it is not necessary to understand the sharpen or `RGB` / `HSV` conversion details - we're just tracing values through the code looking for something unexpected.

Once you've found and fixed the incorrect line of code, remove any breakpoints and re-run from beginning to end to generate the output image, which you can compare with the input. The result should be an image of the cat with correct colors and some sharpening applied.

**Run this cell to see the solution**

When stopped on the return statement, examine the values of `r`, `g`, and `b` (they'll each be in the range of `0..1`). Use `Step Over` again to return from the `hsv_to_rgb()` function and compare the values with those stored in `r_out`, `g_out`, and `b_out` (which are expected to be in the range of `0..255`). You will see that while the `r` value is properly scaled from `0..1` to `0..255`, the `g` and `b` values are not.

**Next: [Lab Summary](summary.ipynb)**